# Fieldkit Project 3: Fine-Tune Multilingual XLM-RoBERTa for Support Ticket Triage

> **Customer Engagement**: Halden Industrie GmbH (Technical Support Operations — Lukas)
> **Objective**: Fine-tune a multi-task `xlm-roberta-base` model on German and English industrial support tickets, calibrate probabilities with temperature scaling, export to ONNX, and quantize to int8 for sub-30ms CPU inference.
> **Hardware Guarantee**: $0 budget using free Kaggle GPU (T4 / P100).

---

### How to Run this Notebook on Kaggle:
1. Go to **kaggle.com** -> **New Notebook**.
2. Accelerator: Select **GPU T4 x2** or **GPU P100** (in Settings panel on right).
3. Upload Datasets: Click **+ Add Data** -> **Upload** -> Drag `data/p3/train.parquet`, `data/p3/val.parquet`, and `data/p3/test.parquet` -> Name dataset `halden-tickets`.
4. Run all cells (approx. 6–8 minutes total runtime).
5. In the **Output** tab, download `fieldkit_xlmr_onnx.zip` and extract to `models/p3/` in your local Fieldkit repo.

In [ ]:
# 1. Environment Setup & Dependencies
!pip install -q transformers datasets accelerate onnx onnxruntime scikit-learn scipy

In [ ]:
# 2. Imports & Reproducibility Seed
import json
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
from onnxruntime.quantization import WeightType, quantize_dynamic
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Active compute device: {DEVICE}')

In [ ]:
# 3. Locate and Ingest Ticket Datasets
KAGGLE_DIR = Path('/kaggle/input/halden-tickets')
LOCAL_DIR = Path('../../data/p3')

if (KAGGLE_DIR / 'train.parquet').exists():
    data_dir = KAGGLE_DIR
elif (LOCAL_DIR / 'train.parquet').exists():
    data_dir = LOCAL_DIR
else:
    matches = list(Path('/kaggle/input').glob('**/train.parquet'))
    data_dir = matches[0].parent if matches else Path('.')

print(f'[*] Ingesting datasets from: {data_dir}')
train_df = pd.read_parquet(data_dir / 'train.parquet')
val_df = pd.read_parquet(data_dir / 'val.parquet')
test_df = pd.read_parquet(data_dir / 'test.parquet')

print(f'[+] Train split: {len(train_df)} tickets ({train_df["language"].value_counts().to_dict()})')
print(f'[+] Val split:   {len(val_df)} tickets')
print(f'[+] Test split:  {len(test_df)} tickets')

In [ ]:
# 4. Taxonomy & Multi-Head Label Encoding
ALLOWED_QUEUES = [
    'technical_defect',
    'order_delivery',
    'returns_rma',
    'billing_inquiry',
    'general_inquiry',
    'other',
]

ALLOWED_PRIORITIES = [
    'low',
    'medium',
    'high',
    'critical',
]

queue2id = {q: i for i, q in enumerate(ALLOWED_QUEUES)}
id2queue = dict(enumerate(ALLOWED_QUEUES))
priority2id = {p: i for i, p in enumerate(ALLOWED_PRIORITIES)}
id2priority = dict(enumerate(ALLOWED_PRIORITIES))

# Class weights for CrossEntropyLoss to address class frequency differences
q_counts = train_df['queue'].map(queue2id).value_counts().sort_index()
q_weights = torch.tensor([len(train_df) / (len(ALLOWED_QUEUES) * max(c, 1)) for c in q_counts], dtype=torch.float32).to(DEVICE)

p_counts = train_df['priority'].map(priority2id).value_counts().sort_index()
p_weights = torch.tensor([len(train_df) / (len(ALLOWED_PRIORITIES) * max(c, 1)) for c in p_counts], dtype=torch.float32).to(DEVICE)

print(f'[*] Queue weights: {q_weights.cpu().numpy().round(3)}')
print(f'[*] Priority weights: {p_weights.cpu().numpy().round(3)}')

In [ ]:
# 5. Tokenization & PyTorch Dataset
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TicketDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 256):
        self.texts = df['text'].tolist()
        self.queues = [queue2id.get(q, queue2id['other']) for q in df['queue']]
        self.priorities = [priority2id.get(p, priority2id['medium']) for p in df['priority']]
        self.languages = df['language'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        enc = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'queue_label': torch.tensor(self.queues[idx], dtype=torch.long),
            'priority_label': torch.tensor(self.priorities[idx], dtype=torch.long),
        }

train_ds = TicketDataset(train_df, tokenizer, max_len=MAX_LEN)
val_ds = TicketDataset(val_df, tokenizer, max_len=MAX_LEN)
test_ds = TicketDataset(test_df, tokenizer, max_len=MAX_LEN)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'[*] DataLoaders initialized: {len(train_loader)} train batches')

In [ ]:
# 6. Multi-Task Model Architecture
class MultiTaskXLMRoberta(nn.Module):
    '''Shared XLM-RoBERTa encoder with dual classification heads.'''
    def __init__(self, model_name: str, num_queues: int = 6, num_priorities: int = 4, dropout: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.queue_head = nn.Linear(hidden_size, num_queues)
        self.priority_head = nn.Linear(hidden_size, num_priorities)

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs[0]
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        pooled = self.dropout(pooled)

        queue_logits = self.queue_head(pooled)
        priority_logits = self.priority_head(pooled)
        return queue_logits, priority_logits

model = MultiTaskXLMRoberta(MODEL_NAME, num_queues=len(ALLOWED_QUEUES), num_priorities=len(ALLOWED_PRIORITIES)).to(DEVICE)
print('[+] MultiTaskXLMRoberta instantiated on', DEVICE)

In [ ]:
# 7. Optimizer, Scheduler & Training Loop (3 Epochs, fp16 on T4)
EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.10

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
)

loss_fn_queue = nn.CrossEntropyLoss(weight=q_weights)
loss_fn_priority = nn.CrossEntropyLoss(weight=p_weights)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

best_val_f1 = -1.0
best_model_weights = None

print('[*] Beginning fine-tuning loop...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        q_labels = batch['queue_label'].to(DEVICE)
        p_labels = batch['priority_label'].to(DEVICE)

        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                q_logits, p_logits = model(input_ids, mask)
                loss = loss_fn_queue(q_logits, q_labels) + loss_fn_priority(p_logits, p_labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            q_logits, p_logits = model(input_ids, mask)
            loss = loss_fn_queue(q_logits, q_labels) + loss_fn_priority(p_logits, p_labels)
            loss.backward()
            optimizer.step()

        scheduler.step()
        total_train_loss += loss.item()

    model.eval()
    all_q_preds, all_q_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            q_labels = batch['queue_label'].to(DEVICE)
            q_logits, _ = model(input_ids, mask)
            all_q_preds.extend(torch.argmax(q_logits, dim=-1).cpu().tolist())
            all_q_true.extend(q_labels.cpu().tolist())

    val_f1 = f1_score(all_q_true, all_q_preds, average='macro', zero_division=0)
    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Queue Macro-F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_model_weights)
print(f'[+] Best model restored with Validation Macro-F1 = {best_val_f1:.4f}')

In [ ]:
# 8. Post-Hoc Temperature Scaling Calibration
model.eval()
val_q_logits, val_q_labels = [], []
val_p_logits, val_p_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        q_logits, p_logits = model(input_ids, mask)
        val_q_logits.append(q_logits.cpu().numpy())
        val_q_labels.append(batch['queue_label'].numpy())
        val_p_logits.append(p_logits.cpu().numpy())
        val_p_labels.append(batch['priority_label'].numpy())

val_q_logits = np.concatenate(val_q_logits, axis=0)
val_q_labels = np.concatenate(val_q_labels, axis=0)
val_p_logits = np.concatenate(val_p_logits, axis=0)
val_p_labels = np.concatenate(val_p_labels, axis=0)

def softmax_np(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

def nll_opt(t, logits, labels):
    probs = softmax_np(logits / max(t, 1e-4))
    n = len(labels)
    return -float(np.mean(np.log(np.clip(probs[np.arange(n), labels], 1e-12, 1.0))))

res_q = minimize_scalar(nll_opt, args=(val_q_logits, val_q_labels), bounds=(0.05, 10.0), method='bounded')
res_p = minimize_scalar(nll_opt, args=(val_p_logits, val_p_labels), bounds=(0.05, 10.0), method='bounded')

temp_q = round(float(res_q.x), 4)
temp_p = round(float(res_p.x), 4)
print(f'[+] Learned Temperature Scalers: Queue T={temp_q}, Priority T={temp_p}')

In [ ]:
# 9. Export PyTorch Model to ONNX
OUT_DIR = Path('/kaggle/working/fieldkit_xlmr_onnx') if Path('/kaggle/working').exists() else Path('models/p3')
OUT_DIR.mkdir(parents=True, exist_ok=True)

dummy_ids = torch.zeros((1, MAX_LEN), dtype=torch.long, device=DEVICE)
dummy_mask = torch.ones((1, MAX_LEN), dtype=torch.long, device=DEVICE)
onnx_path = OUT_DIR / 'model.onnx'

model.eval()
torch.onnx.export(
    model,
    (dummy_ids, dummy_mask),
    str(onnx_path),
    input_names=['input_ids', 'attention_mask'],
    output_names=['queue_logits', 'priority_logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_length'},
        'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
        'queue_logits': {0: 'batch_size'},
        'priority_logits': {0: 'batch_size'},
    },
    opset_version=14,
)
print(f'[+] Exported ONNX model ({onnx_path.stat().st_size / (1024*1024):.2f} MB) -> {onnx_path}')

In [ ]:
# 10. Dynamic Int8 Quantization with ONNX Runtime
quant_path = OUT_DIR / 'model_int8.onnx'
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(quant_path),
    weight_type=WeightType.QInt8,
)
print(f'[+] Quantized Int8 ONNX model ({quant_path.stat().st_size / (1024*1024):.2f} MB) -> {quant_path}')

In [ ]:
# 11. Test Set Evaluation & CPU Latency Benchmark
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 4
ort_sess = ort.InferenceSession(str(quant_path), sess_options, providers=['CPUExecutionProvider'])

test_latencies = []
all_q_preds, all_p_preds = [], []

for _, row in test_df.iterrows():
    t0 = time.perf_counter()
    enc = tokenizer(row['text'], max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='np')
    feeds = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    q_logits, p_logits = ort_sess.run(None, feeds)
    calib_q = softmax_np(q_logits / temp_q)[0]
    calib_p = softmax_np(p_logits / temp_p)[0]
    t_ms = (time.perf_counter() - t0) * 1000.0
    test_latencies.append(t_ms)

    all_q_preds.append(id2queue[int(np.argmax(calib_q))])
    all_p_preds.append(id2priority[int(np.argmax(calib_p))])

test_df['pred_queue'] = all_q_preds
test_df['pred_priority'] = all_p_preds

overall_q_f1 = f1_score(test_df['queue'], test_df['pred_queue'], average='macro', zero_division=0)
en_mask = test_df['language'] == 'en'
de_mask = test_df['language'] == 'de'
en_q_f1 = f1_score(test_df.loc[en_mask, 'queue'], test_df.loc[en_mask, 'pred_queue'], average='macro', zero_division=0)
de_q_f1 = f1_score(test_df.loc[de_mask, 'queue'], test_df.loc[de_mask, 'pred_queue'], average='macro', zero_division=0)
acc = accuracy_score(test_df['queue'], test_df['pred_queue'])

p50 = np.percentile(test_latencies, 50)
p95 = np.percentile(test_latencies, 95)

print('=' * 60)
print('PROJECT 3: XLM-R ONNX INT8 EVALUATION RESULTS')
print(f'Queue Macro-F1 Overall: {overall_q_f1:.4f} (Target: >= 0.85)')
print(f'Queue Macro-F1 EN:      {en_q_f1:.4f}')
print(f'Queue Macro-F1 DE:      {de_q_f1:.4f} (Target: >= 0.82)')
print(f'Queue Accuracy:         {acc*100:.2f}%')
print(f'CPU Latency P50:        {p50:.2f} ms')
print(f'CPU Latency P95:        {p95:.2f} ms (Target: < 30.0 ms)')
print('=' * 60)

In [ ]:
# 12. Save Artifacts Bundle for Local Deployment
tokenizer.save_pretrained(str(OUT_DIR / 'tokenizer'))

labels_meta = {
    'allowed_queues': ALLOWED_QUEUES,
    'allowed_priorities': ALLOWED_PRIORITIES,
    'queue2id': queue2id,
    'id2queue': id2queue,
    'priority2id': priority2id,
    'id2priority': id2priority,
}
with open(OUT_DIR / 'labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels_meta, f, indent=2)

calib_meta = {
    'temperature_queue': temp_q,
    'temperature_priority': temp_p,
    'is_fitted': True,
}
with open(OUT_DIR / 'temperature_scaler.json', 'w', encoding='utf-8') as f:
    json.dump(calib_meta, f, indent=2)

metrics_meta = {
    'queue_macro_f1': round(overall_q_f1, 4),
    'queue_macro_f1_en': round(en_q_f1, 4),
    'queue_macro_f1_de': round(de_q_f1, 4),
    'accuracy': round(acc, 4),
    'latency_p50_ms': round(p50, 2),
    'latency_p95_ms': round(p95, 2),
}
with open(OUT_DIR / 'eval_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_meta, f, indent=2)

# Create zip file for download
zip_path = Path('/kaggle/working/fieldkit_xlmr_onnx.zip') if Path('/kaggle/working').exists() else OUT_DIR / 'fieldkit_xlmr_onnx.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for file_path in OUT_DIR.rglob('*'):
        if file_path.is_file() and not file_path.name.endswith('.zip'):
            z.write(file_path, arcname=file_path.relative_to(OUT_DIR))

print(f'[+] Deployment bundle ready: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)')
print('[!] Download this zip and extract to models/p3/ in your Fieldkit workspace.')